<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_5_2_Bagging_and_Random_Forests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bagging and Random Forests: The Wisdom of Crowds

**Attribution**: Exercises adapted from *Introduction to Modern Statistics (2e)* Chapter 9.6.
Source: [OpenIntro IMS](https://openintro-ims.netlify.app/model-logistic)

**License**: This work is a derivative of [Introduction to Modern Statistics (2e)](https://openintro-ims.netlify.app/) by Mine Çetinkaya-Rundel and Johanna Hardin, used under a [Creative Commons Attribution-ShareAlike 3.0 Unported (CC BY-SA 3.0 US)](https://creativecommons.org/licenses/by-sa/3.0/us/) license.

---

## Introduction: Strength in Numbers

In the previous notebook, we saw that a single decision tree is prone to **overfitting** (high variance). Small changes in the training data can result in vastly different tree structures and predictions.

In this notebook, we solve this instability using **Ensemble Methods**. The core idea is simple: **The wisdom of the crowd.**

Just as averaging the guesses of 1,000 people often yields a better estimate than any single expert, combining many independent (and sometimes noisy) trees creates a model that is remarkably stable and accurate.

### Learning Objectives
By the end of this notebook, you will be able to:
1. **Implement** Bagging to reduce model variance.
2. **Calculate** and interpret Out-of-Bag (OOB) error.
3. **Apply** the Random Forest algorithm to decorrelate individual trees.
4. **Analyze** feature importance to identify key biological markers.
5. **Compare** the stability of single trees vs. ensembles using K-Fold Cross-Validation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
import os

# Set style
sns.set_context("talk")
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load data
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

⚠️ **Note on Execution Time:** This notebook performs cross-validation and trains multiple ensemble models, which can take several minutes to run. Be patient - the computations are worth it for understanding these powerful methods!


---
## 1. Bagging: Fixing Instability

**Bagging** (Bootstrap Aggregating) is a technique designed to reduce the variance of a high-variance estimator (like a Decision Tree).

### How It Works
1.  **Bootstrap Sampling**: We create many random subsets of our training data by sampling *with replacement*. Each subset is the same size as the original dataset, but some samples appear multiple times and others not at all.
2.  **Model Building**: We train a separate, deep Decision Tree on each bootstrap sample.
3.  **Aggregation**: To make a prediction, we take a **majority vote** of all the trees.

### Why It Works
Each individual tree might overfit to the quirks of its specific bootstrap sample. However, because the errors are different across trees, averaging them cancels out the noise, leaving only the true signal.

### Implementation: Bagging with Decision Trees
We will build an ensemble of 500 deep decision trees.

In [ ]:
# Initialize and fit Bagging model
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(), 
    n_estimators=500, 
    oob_score=True, 
    random_state=42
)
bag.fit(X_train, y_train)

print(f"Bagging Accuracy (Test): {accuracy_score(y_test, bag.predict(X_test)):.3f}")
print(f"Out-of-Bag (OOB) Error Score: {bag.oob_score_:.3f}")

### The Out-of-Bag (OOB) Bonus
Notice the `oob_score`. Because we use bootstrap sampling, about **1/3** of our data is left out of each tree. We can treat these "left-out" rows as a test set *during* training. This gives us a validation accuracy estimate **without** needing a separate cross-validation step.

---
## 2. Random Forests: Decorrelating the Trees

Bagging is a powerful improvement, but it has a potential flaw: **Correlation**.

If one feature (e.g., `mean concave points`) is a very strong predictor, *every single tree* in the Bagging ensemble will likely use it as the top split. This makes all the trees look similar. When trees are highly correlated, averaging them doesn't reduce variance as much.

### The Random Forest Twist
**Random Forests** introduce a second layer of randomness:

*   **Feature Subsetting**: At *each split* of *each tree*, the algorithm considers only a **random subset of features** (typically $\sqrt{p}$ features).

This forces trees to consider other, weaker predictors (like `texture` or `smoothness`) that might otherwise be overshadowed. By exploring these alternative paths, the trees become **decorrelated**, and the ensemble becomes much more robust.

In [ ]:
# Initialize and fit Random Forest
rf = RandomForestClassifier(n_estimators=500, random_state=42)
rf.fit(X_train, y_train)

rf_test_acc = accuracy_score(y_test, rf.predict(X_test))
print(f"Random Forest Accuracy (Test): {rf_test_acc:.3f}")

### 3. Feature Importance: Interpreting the Black Box

One major criticism of ensemble methods is that we lose the simple interpretability of a single decision tree. We can no longer just draw a flowchart.

However, Random Forests provide a powerful alternative: **Feature Importance**.

The model tracks how much each feature decreases the **impurity** (Gini score) across all 500 trees. Features that consistently create pure nodes (clean splits) are given high importance scores.

Let's see which biological markers are driving the model's decisions.

In [ ]:
# Plot feature importance
importances = pd.Series(rf.feature_importances_, index=data.feature_names)
top_10 = importances.sort_values(ascending=False).head(10)

top_10.plot(kind='barh')
plt.title("Top 10 Important Features for Breast Cancer Detection")
plt.gca().invert_yaxis()
plt.xlabel("Relative Importance Score")
plt.show()

---
## 4. The Final Verdict: Comparing Stability

We've claimed that ensembles are more stable than single trees. Let's prove it.

We will use **10-Fold Cross-Validation** to compare three models:
1.  **Single Decision Tree** (Deep & Unpruned)
2.  **Bagging Ensemble** (500 Trees)
3.  **Random Forest** (500 Trees + Feature Subsetting)

We expect the Single Tree to have the lowest mean accuracy and the highest standard deviation (instability).

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

# 1. Define Models
models = {
    "Single Tree": DecisionTreeClassifier(random_state=42),
    "Bagging": BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

# 2. Setup Cross-Validation
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 3. Run Comparison
results = {}
print(f"{'Model':<20} | {'Mean Accuracy':<15} | {'Std Dev':<10}")
print("-" * 50)

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')
    results[name] = scores
    print(f"{name:<20} | {scores.mean():.4f}          | {scores.std():.4f}")

### Conclusion

As expected:
*   The **Single Tree** is good but has higher variance (Standard Deviation).
*   **Bagging** significantly improves accuracy and stability.
*   **Random Forest** typically edges out Bagging by further reducing variance through decorrelation.

In modern machine learning, Random Forests (and Boosting, which we will cover next) are almost always preferred over single Decision Trees for tabular data.